# 🫁 Respiratory Sound Analysis

> Analysis of respiratory sounds using advanced signal processing techniques

---

## 📦 Installation & Setup

In [ ]:
!pip install -q kaggle numpy scipy matplotlib pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## ⚙️ Configure Kaggle & Download Dataset

In [ ]:
import os
import shutil
from google.colab import files

KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle.json'
KAGGLE_JSON_LOCAL = os.path.expanduser('~/.kaggle/kaggle.json')

if os.path.exists(KAGGLE_JSON_DRIVE):
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    shutil.copy(KAGGLE_JSON_DRIVE, KAGGLE_JSON_LOCAL)
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
else:
    uploaded = files.upload()
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(KAGGLE_JSON_LOCAL, 'wb') as f:
        f.write(list(uploaded.values())[0])
    os.chmod(KAGGLE_JSON_LOCAL, 0o600)
    shutil.copy(KAGGLE_JSON_LOCAL, KAGGLE_JSON_DRIVE)

In [ ]:
DATASET_PATH = '/content/respiratory_sound_dataset'

print("📥 Downloading dataset from Kaggle...")
!kaggle datasets download -d vbookshelf/respiratory-sound-database
!unzip -q respiratory-sound-database.zip -d {DATASET_PATH}
!rm respiratory-sound-database.zip

print(f"✅ Dataset downloaded")

## 📥 Clone Analysis Repository

In [ ]:
import sys

REPO_PATH = '/content/course_paper'

if os.path.exists(REPO_PATH):
    print(f"✅ Repository already cloned")
else:
    print("📥 Cloning repository...")
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git {REPO_PATH}
    !git -C {REPO_PATH} sparse-checkout set app
    print("✅ Repository cloned")

sys.path.insert(0, REPO_PATH)

In [ ]:
import os
import numpy as np
from scipy.io import wavfile
from pathlib import Path

def load_respiratory_sounds(dataset_path, max_files=20):
    """Load audio files from the dataset efficiently."""
    dataset_path = Path(dataset_path)
    
    if not dataset_path.exists():
        raise FileNotFoundError(f"Path not found: {dataset_path}")
    
    audio_dir = None
    for pattern in ['**/audio_and_txt_files', '**/Respiratory_Sound_Database/**/audio_and_txt_files']:
        matches = list(dataset_path.glob(pattern))
        if matches:
            audio_dir = matches[0]
            break
    
    if not audio_dir or not audio_dir.exists():
        raise FileNotFoundError(f"audio_and_txt_files folder not found in {dataset_path}")
    
    wav_files = sorted(audio_dir.glob("*.wav"))
    load_count = min(max_files, len(wav_files))
    
    signals = {}
    load_errors = 0
    
    for wav_file in wav_files[:load_count]:
        try:
            sample_rate, signal_data = wavfile.read(str(wav_file))
            if signal_data.dtype != np.float64:
                signal_data = signal_data.astype(np.float64)
            if signal_data.ndim > 1:
                signal_data = signal_data[:, 0]
            
            signals[wav_file.name] = {
                'signal': signal_data,
                'sample_rate': sample_rate
            }
        except Exception as e:
            load_errors += 1
            if load_errors <= 3:
                print(f"⚠️ Failed: {wav_file.name}")
    
    if load_errors > 3:
        print(f"⚠️ ...and {load_errors - 3} more errors")
    
    return signals

## 🔊 Extract Features and Create DataFrame

In [ ]:
signals = load_respiratory_sounds(DATASET_PATH, max_files=20)
sample_rate = list(signals.values())[0]['sample_rate']

print(f"✅ Loaded {len(signals)} files @ {sample_rate} Hz")

In [ ]:
import pandas as pd
from pathlib import Path
from app.data_preprocessing import extract_features_batch

diagnosis_path = Path(DATASET_PATH)
diagnosis_files = list(diagnosis_path.rglob('patient_diagnosis.csv'))

if not diagnosis_files:
    raise FileNotFoundError(f"patient_diagnosis.csv not found in {diagnosis_path}")

diagnosis_file = diagnosis_files[0]
print(f"📋 Found: {diagnosis_file}")

diagnosis_df = pd.read_csv(diagnosis_file)
patient_col, diagnosis_col = diagnosis_df.columns[0], diagnosis_df.columns[1]
diagnosis_map = dict(zip(diagnosis_df[patient_col], diagnosis_df[diagnosis_col]))

rows = extract_features_batch(signals, embedding_dim=3, time_delay=1)

for row in rows:
    row['diagnosis'] = diagnosis_map.get(row['patient_id'], 'Unknown')

results_df = pd.DataFrame(rows)

print(f"\n✅ Extracted features: {results_df.shape}")
print(f"\n📋 Diagnoses:\n{results_df['diagnosis'].value_counts()}")
display(results_df.head())

In [ ]:
feature_cols = [
    'low_freq_energy', 'mid_freq_energy', 'high_freq_energy',
    'whistle_strength', 'spectral_centroid', 'peak_frequency',
    'entropy', 'complexity'
]

avg_by_diagnosis = results_df.groupby('diagnosis')[feature_cols].mean().reset_index()

# Add header row with column names
header_row = pd.DataFrame([['diagnosis'] + feature_cols], columns=avg_by_diagnosis.columns)
avg_by_diagnosis_with_header = pd.concat([header_row, avg_by_diagnosis], ignore_index=True)

display(avg_by_diagnosis_with_header)

## 📊 Visualizations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Feature comparison across diagnoses
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, feature in enumerate(feature_cols):
    ax = axes[idx]
    
    diagnoses = avg_by_diagnosis['diagnosis']
    values = avg_by_diagnosis[feature]
    
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(diagnoses)))
    bars = ax.bar(range(len(diagnoses)), values, color=colors, alpha=0.85, edgecolor='black')
    
    # Add value labels
    for i, (bar, val) in enumerate(zip(bars, values)):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    
    ax.set_title(feature.replace('_', ' ').title(), fontsize=10, fontweight='bold')
    ax.set_xticks(range(len(diagnoses)))
    ax.set_xticklabels(diagnoses, rotation=45, ha='right', fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, values.max() * 1.15)

plt.suptitle('Feature Comparison Across Diagnoses', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Energy distribution comparison
energy_cols = ['low_freq_energy', 'mid_freq_energy', 'high_freq_energy']

fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(avg_by_diagnosis))
width = 0.25
colors = ['#2ecc71', '#3498db', '#e74c3c']

for i, energy_col in enumerate(energy_cols):
    values = avg_by_diagnosis[energy_col]
    offset = (i - 1) * width
    bars = ax.bar(x + offset, values, width, label=energy_col.replace('_', ' ').title(),
                  color=colors[i], alpha=0.85, edgecolor='black')
    
    # Add value labels
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Diagnosis', fontsize=12, fontweight='bold')
ax.set_ylabel('Average Normalized Energy', fontsize=12, fontweight='bold')
ax.set_title('Energy Distribution Across Frequency Bands by Diagnosis', 
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(avg_by_diagnosis['diagnosis'], rotation=30, ha='right')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of all features by diagnosis
fig, ax = plt.subplots(figsize=(10, 6))

# Prepare data for heatmap (transpose so diagnoses are rows)
heatmap_data = avg_by_diagnosis.set_index('diagnosis')[feature_cols].T

im = ax.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')

# Set ticks and labels
ax.set_xticks(np.arange(len(heatmap_data.columns)))
ax.set_yticks(np.arange(len(heatmap_data.index)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha='right')
ax.set_yticklabels([col.replace('_', ' ').title() for col in heatmap_data.index])

# Add colorbar
cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Feature Value', rotation=270, labelpad=20)

# Add value annotations
for i in range(len(heatmap_data.index)):
    for j in range(len(heatmap_data.columns)):
        text = ax.text(j, i, f'{heatmap_data.iloc[i, j]:.3f}',
                      ha="center", va="center", color="black", fontsize=8)

ax.set_title('Feature Heatmap by Diagnosis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Time-domain signal comparison by diagnosis
sample_length = 1000  # First 1000 samples

# Collect signals by diagnosis
signals_by_diagnosis = {}
for filename, data in signals.items():
    patient_id = int(filename.split('_')[0])
    diagnosis = diagnosis_map.get(patient_id, 'Unknown')
    
    signal = data['signal'][:sample_length]
    
    # Only include if we have full length
    if len(signal) == sample_length:
        if diagnosis not in signals_by_diagnosis:
            signals_by_diagnosis[diagnosis] = []
        signals_by_diagnosis[diagnosis].append(signal)

# Plot for each diagnosis
diagnoses = avg_by_diagnosis['diagnosis'].tolist()

# Beautiful color palette for each diagnosis
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E2']

for idx, diagnosis in enumerate(diagnoses):
    if diagnosis in signals_by_diagnosis and len(signals_by_diagnosis[diagnosis]) > 0:
        signals_array = np.array(signals_by_diagnosis[diagnosis])
        
        # Calculate mean and std
        mean_signal = np.mean(signals_array, axis=0)
        std_signal = np.std(signals_array, axis=0)
        time_points = np.arange(sample_length)
        
        file_count = results_df[results_df['diagnosis'] == diagnosis].shape[0]
        
        # Use unique color for each diagnosis
        color = colors[idx % len(colors)]
        
        plt.figure(figsize=(12, 4))
        
        # Plot mean line
        plt.plot(time_points, mean_signal, color=color, linewidth=1.5, label='Mean')
        
        # Plot shaded error band (mean ± std)
        plt.fill_between(time_points, 
                        mean_signal - std_signal, 
                        mean_signal + std_signal, 
                        color=color, alpha=0.3, label='±1 SD')
        
        plt.title(f'{diagnosis} - Average Time Domain Signal (n={file_count})', 
                  fontsize=14, fontweight='bold')
        plt.xlabel('Time (samples)', fontsize=12)
        plt.ylabel('Amplitude', fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.show()

In [ ]:
# Entropy vs Complexity scatter plot
fig, ax = plt.subplots(figsize=(10, 8))

# Beautiful color palette for each diagnosis
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A', '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E2']

# Plot each diagnosis as a point
for idx, row in avg_by_diagnosis.iterrows():
    diagnosis = row['diagnosis']
    entropy = row['entropy']
    complexity = row['complexity']
    color = colors[idx % len(colors)]
    
    # Plot point with larger size and edge
    ax.scatter(entropy, complexity, s=300, color=color, alpha=0.85, 
              edgecolors='black', linewidth=2, label=diagnosis, zorder=3)
    
    # Add diagnosis label near the point
    ax.annotate(diagnosis, (entropy, complexity), 
               xytext=(8, 8), textcoords='offset points',
               fontsize=10, fontweight='bold',
               bbox=dict(boxstyle='round,pad=0.5', facecolor=color, alpha=0.3, edgecolor='none'))

ax.set_xlabel('Entropy', fontsize=13, fontweight='bold')
ax.set_ylabel('Complexity', fontsize=13, fontweight='bold')
ax.set_title('Entropy vs Complexity by Diagnosis', fontsize=15, fontweight='bold', pad=20)
ax.grid(True, alpha=0.3, linestyle='--')
ax.legend(loc='best', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()